### Pacotes importados

In [1]:
using LinearAlgebra
using Printf
using Plots


## Chapter 11: Descent methods and line search

### Algorithm 11.2: Initialization of the exact line search

![image.png](attachment:91366503-7153-4bb7-bb9a-fdc1295ffcb9.png)

![image.png](attachment:da12c5e4-ab4f-4d9e-a3b6-b263061d9643.png)

![image.png](attachment:43caf570-a03f-4daa-b814-353c1a71c6b9.png)

Test with Example 11.3: $h(x) = (2+x) \cos(2+x)$.

In [2]:
# Algorithm 11.2: Initialization for exact line search
# Goal: find an interval [a, b] that brackets a local minimum of h.

function bracket_minimum(h; x0=0.0, s=1e-2, k=2.0, max_iter=100)
    a = x0
    ya = h(a)

    b = a + s
    yb = h(b)

    # If function increases, reverse search direction
    if yb > ya
        a, b = b, a
        ya, yb = yb, ya
        s = -s
    end

    for i in 1:max_iter
        c = b + s
        yc = h(c)

        if yc > yb
            return min(a, c), max(a, c)
        end

        a, ya = b, yb
        b, yb = c, yc
        s *= k
    end

    error("Failed to bracket a minimum.")
end

# Example 11.3
h(x) = (2 + x) * cos(2 + x)

a, b = bracket_minimum(h, x0=0.0, s=0.01, k=2.0)

println("Bracket interval found:")
println("a = ", round(a, digits=6))
println("b = ", round(b, digits=6))
println("h(a) = ", round(h(a), digits=6))
println("h(b) = ", round(h(b), digits=6))


Bracket interval found:
a = 0.64
b = 2.56
h(a) = -2.314799
h(b) = -0.692207


### Algorithm 11.3: Exact line search: quadratic interpolation

![image.png](attachment:2e28249c-0c76-4dbf-aced-0ac659ac223b.png)

Test with Example 11.3: $h(x) = (2+x) \cos(2+x)$.

In [3]:
# Algorithm 11.3: Exact line search using quadratic interpolation

function quadratic_interpolation_search(h; a=-2.0, b=2.0, c=4.0, eps=1e-6, max_iter=100)
    x1, x2, x3 = a, b, c

    for k in 1:max_iter
        y1, y2, y3 = h(x1), h(x2), h(x3)

        numerator = y1*(x2^2 - x3^2) + y2*(x3^2 - x1^2) + y3*(x1^2 - x2^2)
        denominator = 2*(y1*(x2 - x3) + y2*(x3 - x1) + y3*(x1 - x2))

        if abs(denominator) < 1e-12
            break
        end

        xbar = numerator / denominator
        ybar = h(xbar)

        if abs(xbar - x2) < eps
            return xbar, ybar, k
        end

        # Keep three points around the best candidate
        points = sort([(x1, y1), (x2, y2), (x3, y3), (xbar, ybar)], by = p -> p[1])
        best_index = argmin([p[2] for p in points])

        if best_index == 1
            selected = points[1:3]
        elseif best_index == length(points)
            selected = points[end-2:end]
        else
            selected = points[best_index-1:best_index+1]
        end

        x1, x2, x3 = selected[1][1], selected[2][1], selected[3][1]
    end

    xs = [x1, x2, x3]
    best = xs[argmin(h.(xs))]
    return best, h(best), max_iter
end

# Example 11.3
h(x) = (2 + x) * cos(2 + x)

# Use the bracket from Algorithm 11.2 and a midpoint
left, right = bracket_minimum(h, x0=0.0, s=0.01, k=2.0)
mid = (left + right) / 2

xmin, hmin, iters = quadratic_interpolation_search(h, a=left, b=mid, c=right)

println("Quadratic interpolation result:")
println("x* = ", round(xmin, digits=6))
println("h(x*) = ", round(hmin, digits=6))
println("iterations = ", iters)


Quadratic interpolation result:
x* = 1.425618
h(x*) = -3.288371
iterations = 8


### Algorithm 11.5: Line search

![image.png](attachment:7ec0a2e5-2f12-4d3b-b3a3-6e490c2c039c.png)

Example 11.2: $f(x) = \frac{1}{2} x_1^2 + \frac{9}{2} x_2^2$.

In [4]:
# Algorithm 11.5: Line search
# Example 11.2: f(x) = 1/2*x1^2 + 9/2*x2^2

f(x) = 0.5*x[1]^2 + 4.5*x[2]^2
grad_f(x) = [x[1], 9*x[2]]

function line_search(f, grad_f, x, d; α=1e-4, β=0.5, t0=1.0, max_iter=100)
    t = t0
    g = grad_f(x)

    for k in 1:max_iter
        if f(x + t*d) <= f(x) + α*t*dot(g, d)
            return t
        end
        t *= β
    end

    return t
end

x0 = [9.0, 1.0]
d = -grad_f(x0)
t = line_search(f, grad_f, x0, d)
xnew = x0 + t*d

println("Line search result:")
println("x0 = ", x0)
println("direction d = ", d)
println("step size t = ", round(t, digits=6))
println("new x = ", round.(xnew, digits=6))
println("f(x0) = ", round(f(x0), digits=6))
println("f(new x) = ", round(f(xnew), digits=6))


Line search result:
x0 = [9.0, 1.0]
direction d = [-9.0, -9.0]
step size t = 0.25
new x = [6.75, -1.25]
f(x0) = 45.0
f(new x) = 29.8125


### Algorithm 11.6: Steepest descent

![image.png](attachment:cb960894-068e-4050-8456-d65868f5be4e.png)

Test in the Rosenbrock function

In [6]:
# Algorithm 11.6: Steepest descent
# Test with the Rosenbrock function

rosenbrock(x) = (1 - x[1])^2 + 100*(x[2] - x[1]^2)^2

function grad_rosenbrock(x)
    return [
        -2*(1 - x[1]) - 400*x[1]*(x[2] - x[1]^2),
        200*(x[2] - x[1]^2)
    ]
end

function steepest_descent(f, grad_f, x0; eps=1e-6, max_iter=10_000, α=1e-4, β=0.5)
    x = copy(x0)
    history = [copy(x)]

    for k in 1:max_iter
        g = grad_f(x)

        if norm(g) < eps
            return x, f(x), k, history
        end

        d = -g
        t = line_search(f, grad_f, x, d, α=α, β=β)
        x = x + t*d
        push!(history, copy(x))
    end

    return x, f(x), max_iter, history
end

x0 = [-1.2, 1.0]
xopt, fopt, iters, history = steepest_descent(rosenbrock, grad_rosenbrock, x0)

println("Steepest descent on Rosenbrock function:")
println("start x0 = ", x0)
println("x* = ", round.(xopt, digits=6))
println("f(x*) = ", round(fopt, digits=8))
println("iterations = ", iters)
println("||grad|| = ", round(norm(grad_rosenbrock(xopt)), digits=8))


Steepest descent on Rosenbrock function:
start x0 = [-1.2, 1.0]
x* = [0.999984, 0.999967]
f(x*) = 0.0
iterations = 10000
||grad|| = 2.313e-5
